In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
class EnsembleMLP(nn.Module):
    """
    Ensemble of your existing SimpleMLP models
    """
    def __init__(self, input_dim=512, output_dim=2, hidden_dims=[256, 128, 64], n_models=10):
        super(EnsembleMLP, self).__init__()
        self.n_models = n_models
        self.models = nn.ModuleList([
            self._create_mlp(input_dim, output_dim, hidden_dims) 
            for _ in range(n_models)
        ])
    
    def _create_mlp(self, input_dim, output_dim, hidden_dims):
        """Create a single MLP model"""
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3)  # Slightly reduced dropout
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        return nn.Sequential(*layers)
    
    def forward(self, x, return_all=False):
        """
        Forward pass through ensemble
        
        Args:
            x: input tensor
            return_all: if True, return predictions from all models
        
        Returns:
            if return_all: tensor of shape [batch_size, n_models, output_dim]
            else: mean prediction [batch_size, output_dim]
        """
        predictions = torch.stack([model(x) for model in self.models], dim=1)
        
        if return_all:
            return predictions
        else:
            return torch.mean(predictions, dim=1)

def train_ensemble(X, y, n_models=10, test_size=0.2, batch_size=128, epochs=100, lr=1e-3, device=None):
    """
    Train ensemble of models
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"Training ensemble of {n_models} models on device: {device}")
    
    # Normalize input features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42
    )
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_tensor = torch.FloatTensor(y_train).to(device)
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_test_tensor = torch.FloatTensor(y_test).to(device)
    
    # Initialize ensemble model
    ensemble = EnsembleMLP(
        input_dim=X.shape[1], 
        output_dim=y.shape[1], 
        n_models=n_models
    ).to(device)
    
    # Train each model in the ensemble separately
    criterion = nn.MSELoss()
    optimizers = [torch.optim.Adam(model.parameters(), lr=lr) for model in ensemble.models]
    
    # Create different bootstrap samples for each model
    n_train = X_train.shape[0]
    
    all_train_losses = []
    all_test_losses = []
    
    for model_idx in range(n_models):
        print(f"\nTraining model {model_idx + 1}/{n_models}")
        
        # Bootstrap sampling for this model
        bootstrap_indices = np.random.choice(n_train, size=n_train, replace=True)
        X_bootstrap = X_train_tensor[bootstrap_indices]
        y_bootstrap = y_train_tensor[bootstrap_indices]
        
        model = ensemble.models[model_idx]
        optimizer = optimizers[model_idx]
        
        train_losses = []
        test_losses = []
        
        for epoch in range(epochs):
            # Training phase
            model.train()
            
            # Create batches
            n_batches = len(X_bootstrap) // batch_size + (1 if len(X_bootstrap) % batch_size != 0 else 0)
            train_loss = 0.0
            
            for i in range(n_batches):
                start_idx = i * batch_size
                end_idx = min((i + 1) * batch_size, len(X_bootstrap))
                
                batch_X = X_bootstrap[start_idx:end_idx]
                batch_y = y_bootstrap[start_idx:end_idx]
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            train_loss /= n_batches
            
            # Test phase
            model.eval()
            with torch.no_grad():
                test_outputs = model(X_test_tensor)
                test_loss = criterion(test_outputs, y_test_tensor).item()
            
            train_losses.append(train_loss)
            test_losses.append(test_loss)
            
            if epoch % 20 == 0:
                print(f"  Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")
        
        all_train_losses.append(train_losses)
        all_test_losses.append(test_losses)
    
    return ensemble, scaler, all_train_losses, all_test_losses

def get_ensemble_predictions(ensemble, X, scaler, device):
    """
    Get predictions and uncertainty estimates from ensemble
    
    Returns:
        mean_pred: mean prediction across ensemble
        epistemic_var: epistemic (model) uncertainty
        predictions_all: all individual model predictions
    """
    ensemble.eval()
    X_scaled = scaler.transform(X)
    X_tensor = torch.FloatTensor(X_scaled).to(device)
    
    with torch.no_grad():
        # Get predictions from all models
        predictions_all = ensemble(X_tensor, return_all=True)  # [batch_size, n_models, output_dim]
        
        # Mean prediction
        mean_pred = torch.mean(predictions_all, dim=1)
        
        # Epistemic uncertainty (variance across models)
        epistemic_var = torch.var(predictions_all, dim=1)
    
    return mean_pred.cpu().numpy(), epistemic_var.cpu().numpy(), predictions_all.cpu().numpy()

def plot_ensemble_results(X_test, y_test, mean_pred, epistemic_var, predictions_all, model_name="Deep Ensemble"):
    """
    Plot ensemble results with uncertainty quantification
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Parameter 0 predictions with epistemic uncertainty
    axes[0, 0].errorbar(y_test[:, 0], mean_pred[:, 0], 
                       yerr=np.sqrt(epistemic_var[:, 0]), 
                       fmt='o', alpha=0.6, markersize=2, capsize=1)
    min_val, max_val = y_test[:, 0].min(), y_test[:, 0].max()
    axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
    axes[0, 0].set_xlabel('True $\Omega_m$')
    axes[0, 0].set_ylabel('Predicted $\Omega_m$')
    axes[0, 0].set_title('$\Omega_m$: True vs Predicted\n(with epistemic uncertainty)')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Parameter 1 predictions with epistemic uncertainty
    axes[0, 1].errorbar(y_test[:, 1], mean_pred[:, 1], 
                       yerr=np.sqrt(epistemic_var[:, 1]), 
                       fmt='o', alpha=0.6, markersize=2, capsize=1)
    min_val, max_val = y_test[:, 1].min(), y_test[:, 1].max()
    axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
    axes[0, 1].set_xlabel('True $\sigma_8$')
    axes[0, 1].set_ylabel('Predicted $\sigma_8$')
    axes[0, 1].set_title('$\sigma_8$: True vs Predicted\n(with epistemic uncertainty)')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Show individual model predictions for a subset of points
    n_show = min(100, len(y_test))
    indices = np.random.choice(len(y_test), n_show, replace=False)
    
    for i in indices[:20]:  # Show first 20 points
        axes[0, 2].plot([y_test[i, 0]] * predictions_all.shape[1], 
                       predictions_all[i, :, 0], 'b-', alpha=0.3)
    axes[0, 2].plot([y_test[indices, 0].min(), y_test[indices, 0].max()], 
                    [y_test[indices, 0].min(), y_test[indices, 0].max()], 'r--')
    axes[0, 2].set_xlabel('True $\Omega_m$')
    axes[0, 2].set_ylabel('Individual Model Predictions')
    axes[0, 2].set_title('Model Disagreement ($\Omega_m$)')
    axes[0, 2].grid(True, alpha=0.3)
    
    # Epistemic uncertainty distributions
    axes[1, 0].hist(np.sqrt(epistemic_var[:, 0]), bins=50, alpha=0.7, label='$\Omega_m$')
    axes[1, 0].hist(np.sqrt(epistemic_var[:, 1]), bins=50, alpha=0.7, label='$\sigma_8$')
    axes[1, 0].set_xlabel('Epistemic Uncertainty (σ)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of Epistemic Uncertainties')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Uncertainty vs prediction accuracy
    errors_0 = np.abs(y_test[:, 0] - mean_pred[:, 0])
    errors_1 = np.abs(y_test[:, 1] - mean_pred[:, 1])
    
    axes[1, 1].scatter(np.sqrt(epistemic_var[:, 0]), errors_0, alpha=0.6, s=10, label='$\Omega_m$')
    axes[1, 1].scatter(np.sqrt(epistemic_var[:, 1]), errors_1, alpha=0.6, s=10, label='$\sigma_8$')
    axes[1, 1].set_xlabel('Predicted Uncertainty (σ)')
    axes[1, 1].set_ylabel('Actual Error')
    axes[1, 1].set_title('Epistemic Uncertainty Calibration')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Parameter space uncertainty visualization
    if len(y_test) > 1000:  # Subsample for visualization
        indices = np.random.choice(len(y_test), 1000, replace=False)
        y_subset = y_test[indices]
        var_subset = epistemic_var[indices]
    else:
        y_subset = y_test
        var_subset = epistemic_var
    
    # Use total uncertainty (sum of both parameter uncertainties) for color
    total_uncertainty = np.sqrt(var_subset[:, 0] + var_subset[:, 1])
    scatter = axes[1, 2].scatter(y_subset[:, 0], y_subset[:, 1], 
                                c=total_uncertainty, cmap='viridis', 
                                alpha=0.6, s=10)
    axes[1, 2].set_xlabel('True $\Omega_m$')
    axes[1, 2].set_ylabel('True $\sigma_8$')
    axes[1, 2].set_title('Parameter Space\n(colored by epistemic uncertainty)')
    plt.colorbar(scatter, ax=axes[1, 2])
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.suptitle(f'{model_name} - Epistemic Uncertainty Quantification')
    plt.tight_layout()
    plt.show()

In [ ]:
# Example usage:
ensemble, scaler, train_losses, test_losses = train_ensemble(
    np.asarray(TNG_features[0]), 
    np.asarray(TNG_cosmo_params[0]),
    n_models=10
)

# # Get predictions and uncertainties
mean_pred, epistemic_var, predictions_all = get_ensemble_predictions(
    ensemble, X_test, scaler, device
)

plot_ensemble_results(X_test, y_test, mean_pred, epistemic_var, predictions_all)